# Part 2: CLIP Text and Image Embeddings
Loads a pretrained CLIP model, tokenizes captions from the curated COCO subset, computes image and text embeddings for a sample of pairs, and checks that the pipeline runs end to end.

## 1. Setup

In [ ]:
# Find the repo, move into the notebook folder, and pick a device
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/rasmussenj03/IE7615-Group3-Project2.git"
WORK_DIR = "Milestone 1"  # folder holding the notebooks, relative to the repo root
COLAB_USE_DRIVE = True  # False -> clone into /content (outputs are lost when the runtime ends)
COLAB_REPO_DIR = ("/content/drive/MyDrive/IE7615-Group3-Project2" if COLAB_USE_DRIVE
                  else "/content/IE7615-Group3-Project2")

if "google.colab" in sys.modules:
    if COLAB_USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    if not Path(COLAB_REPO_DIR, "requirements.txt").is_file():
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")  # Colab secret holding a GitHub access token
        clone = subprocess.run(["git", "clone", "-q", REPO_URL.replace("https://", f"https://{token}@"),
                                COLAB_REPO_DIR], capture_output=True)
        if clone.returncode != 0:
            raise RuntimeError("git clone failed - check the GITHUB_TOKEN secret and repo access")
        subprocess.run(["git", "-C", COLAB_REPO_DIR, "remote", "set-url", "origin", REPO_URL])
    root = Path(COLAB_REPO_DIR)
    # Install the pinned packages (local runs use the .venv instead)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")],
                   check=True)
else:
    root = Path.cwd()
    while not (root / "requirements.txt").exists() and root != root.parent:
        root = root.parent

PROJECT_ROOT = root
os.chdir(PROJECT_ROOT / WORK_DIR)

import torch
if torch.cuda.is_available():
    device = "cuda"
    device_name = "cuda - " + torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    device = device_name = "mps"
else:
    device = device_name = "cpu"
print("Project root:", PROJECT_ROOT)
print("Working folder:", Path.cwd())
print("PyTorch", torch.__version__, "| device:", device_name)

In [ ]:
# Imports
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

In [ ]:
# Seed for reproducible sampling
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("transformers", transformers.__version__)

## 2. Configuration

In [ ]:
# Paths and settings
DATA_DIR = Path("coco_export")
LABELS_PATH = DATA_DIR / "labels.json"
OUTPUT_DIR = Path("outputs") / "clip"

MODEL_ID = "openai/clip-vit-base-patch32"
N_SAMPLES = 256
BATCH_SIZE = 32

## 3. Load image–caption pairs

In [ ]:
# Read the labels exported by the dataset notebook
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

print("Records:", len(records))
print("Example:", records[0])

In [ ]:
# Sample pairs, keeping the first caption of each image
sample = random.sample(records, min(N_SAMPLES, len(records)))

image_paths = [DATA_DIR / r["image"] for r in sample]
captions = [r["captions"][0] for r in sample]

missing = [p for p in image_paths if not p.exists()]
print("Sampled pairs:", len(sample))
print("Missing image files:", len(missing))

In [ ]:
# Preview a few pairs
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, path, caption in zip(axes, image_paths[:4], captions[:4]):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(caption, fontsize=9, wrap=True)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Load CLIP

In [ ]:
# Load the pretrained model and its processor
model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e6:.1f}M")
print("Embedding size:", model.config.projection_dim)

## 5. Tokenize captions

In [ ]:
# Tokenize one caption to inspect the token IDs
example = processor.tokenizer(captions[0])
print("Caption:", captions[0])
print("Token IDs:", example["input_ids"])
print("Tokens:", processor.tokenizer.convert_ids_to_tokens(example["input_ids"]))

In [ ]:
# Check caption lengths against CLIP's 77-token limit
max_len = processor.tokenizer.model_max_length
lengths = [len(processor.tokenizer(c)["input_ids"]) for c in captions]

print("Max tokens allowed:", max_len)
print("Longest caption (tokens):", max(lengths))
print("Captions over the limit:", sum(n > max_len for n in lengths))

## 6. Compute embeddings

In [ ]:
# Embed images in batches and normalize to unit length
@torch.no_grad()
def embed_images(paths):
    chunks = []
    for i in range(0, len(paths), BATCH_SIZE):
        images = [Image.open(p).convert("RGB") for p in paths[i:i + BATCH_SIZE]]
        inputs = processor(images=images, return_tensors="pt").to(device)
        feats = model.get_image_features(**inputs)
        chunks.append(torch.nn.functional.normalize(feats, dim=-1).cpu())
    return torch.cat(chunks)

In [ ]:
# Embed captions in batches and normalize to unit length
@torch.no_grad()
def embed_texts(texts):
    chunks = []
    for i in range(0, len(texts), BATCH_SIZE):
        inputs = processor.tokenizer(
            texts[i:i + BATCH_SIZE],
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)
        feats = model.get_text_features(**inputs)
        chunks.append(torch.nn.functional.normalize(feats, dim=-1).cpu())
    return torch.cat(chunks)

In [ ]:
# Run both encoders
image_embeds = embed_images(image_paths)
text_embeds = embed_texts(captions)

print("Image embeddings:", tuple(image_embeds.shape))
print("Text embeddings:", tuple(text_embeds.shape))

## 7. Verify the pipeline

In [ ]:
# Basic checks on shape, finiteness, and normalization
assert image_embeds.shape == text_embeds.shape
assert torch.isfinite(image_embeds).all() and torch.isfinite(text_embeds).all()

print("Image norms:", image_embeds.norm(dim=-1).min().item(), "to", image_embeds.norm(dim=-1).max().item())
print("Text norms:", text_embeds.norm(dim=-1).min().item(), "to", text_embeds.norm(dim=-1).max().item())

In [ ]:
# Cosine similarity between every image and every caption
similarity = image_embeds @ text_embeds.T
n = similarity.shape[0]

matched = similarity.diag().mean().item()
unmatched = similarity[~torch.eye(n, dtype=torch.bool)].mean().item()
print(f"Mean similarity, matching pairs:     {matched:.3f}")
print(f"Mean similarity, non-matching pairs: {unmatched:.3f}")

In [ ]:
# Retrieval accuracy: is the true caption (or image) ranked in the top k?
def recall_at_k(sim, k):
    topk = sim.topk(k, dim=1).indices
    targets = torch.arange(sim.shape[0]).unsqueeze(1)
    return (topk == targets).any(dim=1).float().mean().item()

print(f"Chance level (top 1): {1 / n:.3f}")
for k in (1, 5):
    print(f"Image→text R@{k}: {recall_at_k(similarity, k):.3f}   "
          f"Text→image R@{k}: {recall_at_k(similarity.T, k):.3f}")

In [ ]:
# Heatmap of the first 16 pairs; a bright diagonal means matching pairs score highest
k = min(16, n)
plt.figure(figsize=(6, 5))
plt.imshow(similarity[:k, :k].numpy(), cmap="viridis")
plt.colorbar(label="Cosine similarity")
plt.xlabel("Caption index")
plt.ylabel("Image index")
plt.title("Image–caption similarity")
plt.show()

In [ ]:
# Top 3 retrieved captions for a few images
for i in range(3):
    top = similarity[i].topk(3)
    print(f"\nImage {image_paths[i].name} | true caption: {captions[i]}")
    for score, j in zip(top.values.tolist(), top.indices.tolist()):
        mark = "✓" if j == i else " "
        print(f"  {mark} {score:.3f}  {captions[j]}")

## 8. Save embeddings

In [ ]:
# Save embeddings with the paths and captions they belong to
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / "clip_embeddings.pt"

torch.save({
    "model_id": MODEL_ID,
    "image_paths": [str(p) for p in image_paths],
    "captions": captions,
    "image_embeds": image_embeds,
    "text_embeds": text_embeds,
}, out_path)

print("Saved to", out_path)

In [ ]:
# Reload to confirm the file is usable
saved = torch.load(out_path)
assert torch.equal(saved["image_embeds"], image_embeds)
print("Reloaded:", len(saved["captions"]), "pairs,", tuple(saved["text_embeds"].shape))